In [1]:
import pandas as pd
import numpy as np
import torch

from src.forecaster import Forecaster
from src.models import LSTMHistoric, LSTMEncoderDecoder
from src.data import DataSource
from src.utils.scores_and_losses import MAE, RMSE, DILATE

In [ ]:
HISTORIC_COLS = ['prcp', 'tmax', 'tmin', 'vp', 'srad']
FUTURE_COLS   = ['prcp']
TARGET_COL    = 'target'
HORIZON       = 7
SEQ_LEN       = 30
NODATA_VALUES = [-999]

train_source = DataSource(
    csv='data/data.csv',
    start='1980-01-01',
    end='2004-12-31',
    csv_index_col=1,
    nodata_values=NODATA_VALUES,
)
val_source = DataSource(
    csv='data/data.csv',
    start='2005-01-01',
    end='2009-12-31',
    csv_index_col=1,
    nodata_values=NODATA_VALUES,
)
test_source = DataSource(
    csv='data/data.csv',
    start='2010-01-01',
    end='2014-12-31',
    csv_index_col=1,
    nodata_values=NODATA_VALUES,
)

In [21]:
fc = Forecaster(
    model=LSTMHistoric,
    model_config={
        'LSTM_hidden_size': 16,
        'LSTM_num_layers': 1,
        'dropout_rate': 0.0,
    },
    name='LSTMHistoric_test',
    training_datasets=[train_source],
    validation_datasets=[val_source],
    historic_cols=HISTORIC_COLS,
    # future_cols=FUTURE_COLS,
    target_col=TARGET_COL,
    forecasting_horizon=HORIZON,
    historic_input_sequence_length=SEQ_LEN,
    loss_function='mae',
    validation_score='nse',
    validation_logging_criteria=['rmse'],
    minimize_validation_score=False,
    save_path='models',
    batch_size=256,
    number_of_epochs=50,
    use_torch_compile=False,
    save_weights_every_n_epochs=1,
)

print(f'LSTMHistoric parameters: {fc.compute_number_of_parameters()}')

LSTMHistoric parameters: 1591


In [22]:
fc.fit()

Best model saved → models/LSTMHistoric_test/model_LSTMHistoric_test_best.pt
Checkpoint saved  → models/LSTMHistoric_test/model_LSTMHistoric_test_epoch_1.pt
Epoch 1/50  train_loss=0.6923  val_loss=0.6471  val_score=-0.4670  best=-0.4670
Best model saved → models/LSTMHistoric_test/model_LSTMHistoric_test_best.pt
Checkpoint saved  → models/LSTMHistoric_test/model_LSTMHistoric_test_epoch_2.pt
Epoch 2/50  train_loss=0.5792  val_loss=0.5120  val_score=-0.0240  best=-0.0240
Best model saved → models/LSTMHistoric_test/model_LSTMHistoric_test_best.pt
Checkpoint saved  → models/LSTMHistoric_test/model_LSTMHistoric_test_epoch_3.pt
Epoch 3/50  train_loss=0.4764  val_loss=0.4277  val_score=0.1998  best=0.1998
Best model saved → models/LSTMHistoric_test/model_LSTMHistoric_test_best.pt
Checkpoint saved  → models/LSTMHistoric_test/model_LSTMHistoric_test_epoch_4.pt
Epoch 4/50  train_loss=0.4036  val_loss=0.3587  val_score=0.3812  best=0.3812
Best model saved → models/LSTMHistoric_test/model_LSTMHistor

In [23]:
import matplotlib.pyplot as plt
from src.data.timeseries_dataset import TimeSeriesDataset
from src.data.normalization import _read_1d_df

# predictions via predict() itself
preds = fc.predict(test_source, batch_size=256)
y_pred = preds.squeeze(-1).detach().numpy()  # (n_windows, horizon)

# true values + dates: target_col doesn't affect which windows get built (only x_h/x_f NaN
# does), so this dataset covers the exact same windows as predict()'s internal one — it's
# only built here to read off the (source_idx, t) index for alignment, not to run the model
test_ds = TimeSeriesDataset(
    sources=[test_source],
    seq_len=SEQ_LEN,
    horizon=HORIZON,
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    norm_stats=fc.norm_stats,
    target_col=TARGET_COL,
)
test_df = _read_1d_df(test_source)
t_values = np.array([t for _, t in test_ds._index])
y_true = np.stack([test_df[TARGET_COL].values[t + SEQ_LEN : t + SEQ_LEN + HORIZON] for t in t_values])

def lead_dates(lead):
    return test_df.index[t_values + SEQ_LEN + (lead - 1)]

print(f'Evaluated {y_true.shape[0]} windows over a {y_true.shape[1]}-day horizon on test_source')

Evaluated 1789 windows over a 7-day horizon on test_source


In [24]:
fc.val_log_aggregated['mean']

array([(0.64707094, -0.46696874, 3221.1323),
       (0.5120442 , -0.02398851, 3087.375 ),
       (0.42773727,  0.19981207, 2775.899 ),
       (0.35866946,  0.38121983, 2389.4478),
       (0.31592074,  0.4947028 , 2105.812 ),
       (0.28993723,  0.55116314, 1912.16  ),
       (0.26319918,  0.60707515, 1731.8143),
       (0.2572349 ,  0.61644375, 1680.3174),
       (0.2523121 ,  0.6255299 , 1633.8328),
       (0.24072123,  0.64184827, 1562.7207),
       (0.24551037,  0.6396056 , 1574.1823),
       (0.23996784,  0.6533182 , 1536.4147),
       (0.2460973 ,  0.63814884, 1576.4768),
       (0.2402055 ,  0.6465969 , 1541.7472),
       (0.24064875,  0.6471678 , 1552.4789),
       (0.24273933,  0.63802797, 1558.0187),
       (0.243697  ,  0.6361014 , 1548.8105),
       (0.24988148,  0.63084733, 1594.0349),
       (0.24496081,  0.6362949 , 1556.6136),
       (0.25367135,  0.61998737, 1620.8206),
       (0.24730341,  0.6328316 , 1580.9255),
       (0.24972221,  0.6252374 , 1599.7972),
       (0.

In [ ]:
fc2 = Forecaster(
    model=LSTMEncoderDecoder,
    model_config={
        'LSTM_hidden_size': 16,
        'LSTM_num_layers': 1,
        'dropout_rate': 0.0,
        'encoder_downscale_layer_size': 8,
    },
    name='LSTMEncoderDecoder_test',
    training_datasets=[train_source],
    validation_datasets=[val_source],
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    target_col=TARGET_COL,
    forecasting_horizon=HORIZON,
    historic_input_sequence_length=SEQ_LEN,
    loss_function='mae',
    validation_score='nse',
    validation_logging_criteria=['rmse'],
    minimize_validation_score=False,
    save_path='models',
    batch_size=256,
    number_of_epochs=50,
    use_torch_compile=False,
    save_weights_every_n_epochs=1,
)

print(f'LSTMEncoderDecoder parameters: {fc2.compute_number_of_parameters()}')

In [ ]:
fc2.fit()

In [ ]:
preds2 = fc2.predict(test_source, batch_size=256)
y_pred2 = preds2.squeeze(-1).detach().numpy()  # (n_windows, horizon)

# same pattern as fc's eval cell above — small TimeSeriesDataset built with fc2's own
# future_cols, purely for window/date alignment via _index, not for running the model
test_ds2 = TimeSeriesDataset(
    sources=[test_source],
    seq_len=SEQ_LEN,
    horizon=HORIZON,
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    norm_stats=fc2.norm_stats,
    target_col=TARGET_COL,
)
t_values2 = np.array([t for _, t in test_ds2._index])
y_true2 = np.stack([test_df[TARGET_COL].values[t + SEQ_LEN : t + SEQ_LEN + HORIZON] for t in t_values2])

def lead_dates2(lead):
    return test_df.index[t_values2 + SEQ_LEN + (lead - 1)]

print(f'Evaluated {y_true2.shape[0]} windows over a {y_true2.shape[1]}-day horizon on test_source (LSTMEncoderDecoder)')

In [ ]:
epochs = np.arange(1, len(fc.loss_log) + 1)
epochs2 = np.arange(1, len(fc2.loss_log) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs, fc.loss_log, color='tab:blue', label='LSTMHistoric train')
axes[0].plot(epochs, fc.val_log_aggregated['mean']['loss'], color='tab:blue', linestyle='--', label='LSTMHistoric val')
axes[0].plot(epochs2, fc2.loss_log, color='tab:red', label='LSTMEncoderDecoder train')
axes[0].plot(epochs2, fc2.val_log_aggregated['mean']['loss'], color='tab:red', linestyle='--', label='LSTMEncoderDecoder val')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('MAE (normalized)')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs, fc.val_log_aggregated['mean']['val_score'], color='tab:blue', label='LSTMHistoric')
axes[1].plot(epochs2, fc2.val_log_aggregated['mean']['val_score'], color='tab:red', label='LSTMEncoderDecoder')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('NSE')
axes[1].set_title('Validation score (NSE)')
axes[1].legend()

axes[2].plot(epochs, fc.val_log_aggregated['mean']['RMSE'], color='tab:blue', label='LSTMHistoric')
axes[2].plot(epochs2, fc2.val_log_aggregated['mean']['RMSE'], color='tab:red', label='LSTMEncoderDecoder')
axes[2].set_xlabel('epoch')
axes[2].set_ylabel('RMSE')
axes[2].set_title('Validation RMSE')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
LEADS_TO_SHOW = [1, 4, 7]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(lead_dates(1), y_true[:, 0], color='black', lw=1, label='observed')
colors = plt.cm.viridis(np.linspace(0, 0.8, len(LEADS_TO_SHOW)))
for lead, c in zip(LEADS_TO_SHOW, colors):
    ax.plot(lead_dates(lead), y_pred[:, lead - 1], color=c, lw=1, alpha=0.8, linestyle='-', label=f'LSTMHistoric, lead={lead}d')
    ax.plot(lead_dates2(lead), y_pred2[:, lead - 1], color=c, lw=1, alpha=0.8, linestyle='--', label=f'LSTMEncoderDecoder, lead={lead}d')
ax.set_xlabel('date')
ax.set_ylabel(TARGET_COL)
ax.set_title('Hydrograph — observed vs. predicted')
ax.legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
LEADS_TO_SHOW = [1, 4, 7]

fig, axes = plt.subplots(1, len(LEADS_TO_SHOW), figsize=(5 * len(LEADS_TO_SHOW), 5))
lims = [0, np.nanmax([np.nanmax(y_true), np.nanmax(y_pred), np.nanmax(y_true2), np.nanmax(y_pred2)])]

for ax, lead in zip(axes, LEADS_TO_SHOW):
    ax.scatter(y_true[:, lead - 1], y_pred[:, lead - 1], s=8, alpha=0.4, color='tab:blue', label='LSTMHistoric')
    ax.scatter(y_true2[:, lead - 1], y_pred2[:, lead - 1], s=8, alpha=0.4, color='tab:red', label='LSTMEncoderDecoder')
    ax.plot(lims, lims, color='black', lw=1, linestyle='--', label='1:1')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel('observed')
    ax.set_ylabel(f'predicted (lead={lead}d)')
    ax.set_title(f'Observed vs. predicted — lead={lead}d')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
LEADS_TO_SHOW = [1, 4, 7]

def flow_duration_curve(values):
    values = values[~np.isnan(values)]
    sorted_vals = np.sort(values)[::-1]
    exceedance = np.arange(1, len(sorted_vals) + 1) / (len(sorted_vals) + 1) * 100
    return exceedance, sorted_vals

fig, axes = plt.subplots(1, len(LEADS_TO_SHOW), figsize=(6 * len(LEADS_TO_SHOW), 5), sharey=True)

for ax, lead in zip(axes, LEADS_TO_SHOW):
    exc_obs, sorted_obs = flow_duration_curve(y_true[:, lead - 1])
    exc_pred, sorted_pred = flow_duration_curve(y_pred[:, lead - 1])
    exc_pred2, sorted_pred2 = flow_duration_curve(y_pred2[:, lead - 1])
    ax.plot(exc_obs, sorted_obs, label='observed', color='black')
    ax.plot(exc_pred, sorted_pred, label='LSTMHistoric', color='tab:blue')
    ax.plot(exc_pred2, sorted_pred2, label='LSTMEncoderDecoder', color='tab:red')
    ax.set_yscale('log')
    ax.set_xlabel('exceedance probability (%)')
    ax.set_title(f'Flow duration curve — lead={lead}d')
    ax.legend()

axes[0].set_ylabel(f'{TARGET_COL} (log scale)')
plt.tight_layout()
plt.show()

In [ ]:
from src.utils.scores_and_losses import NSE

_nse_metric = NSE()

def nse(obs, pred):
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    return _nse_metric(torch.tensor(pred[mask]), torch.tensor(obs[mask])).item()

leads = np.arange(1, HORIZON + 1)
mae_per_lead = [np.nanmean(np.abs(y_true[:, i] - y_pred[:, i])) for i in range(HORIZON)]
nse_per_lead = [nse(y_true[:, i], y_pred[:, i]) for i in range(HORIZON)]
mae_per_lead2 = [np.nanmean(np.abs(y_true2[:, i] - y_pred2[:, i])) for i in range(HORIZON)]
nse_per_lead2 = [nse(y_true2[:, i], y_pred2[:, i]) for i in range(HORIZON)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(leads, mae_per_lead, marker='o', label='LSTMHistoric')
axes[0].plot(leads, mae_per_lead2, marker='o', label='LSTMEncoderDecoder')
axes[0].set_xlabel('lead time (days)')
axes[0].set_ylabel('MAE')
axes[0].set_title('MAE vs. lead time')
axes[0].legend()

axes[1].plot(leads, nse_per_lead, marker='o', label='LSTMHistoric')
axes[1].plot(leads, nse_per_lead2, marker='o', label='LSTMEncoderDecoder')
axes[1].set_xlabel('lead time (days)')
axes[1].set_ylabel('NSE')
axes[1].set_title('NSE vs. lead time')
axes[1].legend()

plt.tight_layout()
plt.show()